# Pydantic for AI Agents

This notebook teaches Pydantic from the basics and then connects it directly to AI-agent development.

You will learn:

- `BaseModel`
- validation and type conversion
- optional fields and defaults
- lists, dictionaries, and nested models
- enums
- `Field` constraints
- custom validators
- JSON and dictionary serialization
- JSON Schema generation
- structured LLM output
- tool input/output schemas
- agent decisions
- agent state
- guardrails
- a small end-to-end mock agent

This notebook uses **Pydantic v2**.

## 1. Why Pydantic matters in AI agents

Agents constantly move data between components:

```text
User -> LLM -> Tool -> Database/API -> LLM -> Answer
```

LLMs naturally produce text, but software needs predictable structures.

For example, an LLM might return:

```json
{
  "city": "Geneva",
  "days": "five"
}
```

Your application may require:

```python
city: str
days: int
```

Pydantic acts as a validation and schema layer between uncertain model output and deterministic software.

## 2. Installation

If needed:

```bash
pip install pydantic
```

In [ ]:
import pydantic
print("Pydantic version:", pydantic.__version__)

## 3. First Pydantic model

A Pydantic model inherits from `BaseModel`.

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="Meisam", age=35)
print(user)

Pydantic uses Python type annotations as a schema.

Here:

```python
name: str
age: int
```

means those fields are expected to contain a string and an integer.

## 4. Automatic type conversion

Pydantic can often convert compatible values.

In [ ]:
user = User(name="Meisam", age="35")

print(user)
print(type(user.age))

The string `"35"` is converted to the integer `35`.

This is useful when data comes from JSON, APIs, files, tools, or LLM output.

## 5. Validation errors

In [ ]:
from pydantic import ValidationError

try:
    User(name="Meisam", age="not_a_number")
except ValidationError as e:
    print(e)

This is one of the main benefits for AI-agent systems: malformed data fails early instead of quietly poisoning later steps.

## 6. Default values

In [ ]:
class AgentConfig(BaseModel):
    model_name: str = "local-llm"
    temperature: float = 0.2
    max_iterations: int = 5

config = AgentConfig()
print(config)

In [ ]:
config = AgentConfig(temperature=0.7, max_iterations=10)
print(config)

## 7. Optional fields

In [ ]:
class ResearchRequest(BaseModel):
    topic: str
    year: int | None = None
    author: str | None = None

request = ResearchRequest(topic="Quantum machine learning")
print(request)

Optional fields are common in agent systems because user requests are often incomplete.

## 8. Lists and dictionaries

In [ ]:
class SearchResult(BaseModel):
    title: str
    authors: list[str]
    metadata: dict[str, str]

result = SearchResult(
    title="Quantum ML for Jets",
    authors=["Alice", "Bob"],
    metadata={"year": "2026", "journal": "JHEP"}
)

print(result)

## 9. Nested models

In [ ]:
class Author(BaseModel):
    name: str
    institution: str

class Paper(BaseModel):
    title: str
    year: int
    authors: list[Author]

paper = Paper(
    title="Quantum Methods for Jet Classification",
    year=2026,
    authors=[
        {"name": "Alice", "institution": "CERN"},
        {"name": "Bob", "institution": "DESY"}
    ]
)

print(paper)
print(type(paper.authors[0]))

Pydantic automatically converts the nested dictionaries into `Author` objects.

## 10. Enums for controlled choices

Agents often need to choose one action from a known set.

In [ ]:
from enum import Enum

class ToolName(str, Enum):
    WEB_SEARCH = "web_search"
    DATABASE = "database"
    CALCULATOR = "calculator"

class ToolCall(BaseModel):
    tool: ToolName
    query: str

call = ToolCall(tool="web_search", query="quantum computing papers")
print(call)

In [ ]:
try:
    ToolCall(tool="magic_tool", query="do something")
except ValidationError as e:
    print(e)

Using enums prevents an agent from inventing arbitrary tool names.

## 11. Field constraints

In [ ]:
from pydantic import Field

class LLMConfig(BaseModel):
    temperature: float = Field(default=0.2, ge=0.0, le=2.0)
    max_tokens: int = Field(default=1000, gt=0, le=10000)

config = LLMConfig(temperature=0.7, max_tokens=2000)
print(config)

In [ ]:
try:
    LLMConfig(temperature=5, max_tokens=-10)
except ValidationError as e:
    print(e)

## 12. Custom field validators

In [ ]:
from pydantic import field_validator

class ScientificQuery(BaseModel):
    topic: str
    max_papers: int

    @field_validator("topic")
    @classmethod
    def topic_must_not_be_empty(cls, value):
        if not value.strip():
            raise ValueError("topic cannot be empty")
        return value

    @field_validator("max_papers")
    @classmethod
    def limit_papers(cls, value):
        if value > 100:
            raise ValueError("max_papers cannot exceed 100")
        return value

query = ScientificQuery(topic="Particle transformers", max_papers=10)
print(query)

## 13. Convert a model to a dictionary

In [ ]:
user = User(name="Meisam", age=35)

data = user.model_dump()

print(data)
print(type(data))

## 14. Convert a model to JSON

In [ ]:
json_text = user.model_dump_json(indent=2)
print(json_text)

## 15. Validate a Python dictionary

In [ ]:
raw_data = {
    "name": "Meisam",
    "age": "35"
}

validated = User.model_validate(raw_data)
print(validated)

## 16. Validate JSON text

This is especially useful for structured LLM output.

In [ ]:
raw_json = '''
{
  "name": "Alice",
  "age": 32
}
'''

validated = User.model_validate_json(raw_json)
print(validated)

## 17. JSON Schema generation

In [ ]:
schema = User.model_json_schema()
schema

JSON Schema is extremely important in AI-agent tooling.

Frameworks and model providers can use it to tell the model exactly which fields, types, and constraints are expected.

# Part II: Pydantic in AI Agents

## 18. Structured LLM output

Suppose an LLM must extract information from text.

Instead of accepting arbitrary prose, define the required structure.

In [ ]:
class PersonExtraction(BaseModel):
    name: str
    age: int
    occupation: str

In [ ]:
llm_output = '''
{
  "name": "Alice",
  "age": 32,
  "occupation": "Physicist"
}
'''

person = PersonExtraction.model_validate_json(llm_output)
print(person)

The conceptual pipeline is:

```text
User text
   |
   v
  LLM
   |
   v
Structured JSON
   |
   v
Pydantic validation
   |
   +--> valid --> application
   |
   +--> invalid --> retry / repair / reject
```

## 19. Tool input schemas

Suppose the agent can search arXiv.

In [ ]:
class ArxivSearchInput(BaseModel):
    query: str
    max_results: int = Field(default=5, ge=1, le=20)
    year_from: int | None = None

ArxivSearchInput.model_json_schema()

In [ ]:
raw_tool_call = {
    "query": "quantum machine learning jets",
    "max_results": 10,
    "year_from": 2024
}

tool_args = ArxivSearchInput.model_validate(raw_tool_call)
print(tool_args)

## 20. Connect the validated schema to a Python tool

In [ ]:
def search_arxiv(arguments: ArxivSearchInput):
    print("Query:", arguments.query)
    print("Max results:", arguments.max_results)
    print("Year from:", arguments.year_from)

    return [
        "Paper A",
        "Paper B",
        "Paper C"
    ]

results = search_arxiv(tool_args)
print(results)

## 21. Structured tool output

In [ ]:
class PaperResult(BaseModel):
    title: str
    year: int
    score: float

class ArxivSearchOutput(BaseModel):
    query: str
    papers: list[PaperResult]

output = ArxivSearchOutput(
    query="quantum machine learning jets",
    papers=[
        {
            "title": "Quantum Jet Classification",
            "year": 2026,
            "score": 0.94
        },
        {
            "title": "Hybrid QML for LHC Physics",
            "year": 2025,
            "score": 0.89
        }
    ]
)

print(output)

Now the whole tool interface has a contract:

```text
LLM
 |
 v
Tool Input Model
 |
 v
Python Tool
 |
 v
Tool Output Model
 |
 v
LLM
```

## 22. Structured agent decisions

In [ ]:
class AgentAction(str, Enum):
    SEARCH = "search"
    CALCULATE = "calculate"
    ANSWER = "answer"

class AgentDecision(BaseModel):
    action: AgentAction
    reason: str
    query: str | None = None

decision = AgentDecision(
    action="search",
    reason="Recent information is required.",
    query="QML particle physics 2026"
)

print(decision)

An invalid action will be rejected:

In [ ]:
try:
    AgentDecision(
        action="invent_a_new_tool",
        reason="The model felt adventurous."
    )
except ValidationError as e:
    print(e)

## 23. Agent state

In [ ]:
class AgentState(BaseModel):
    user_query: str
    retrieved_documents: list[str] = Field(default_factory=list)
    tool_calls: list[str] = Field(default_factory=list)
    final_answer: str | None = None
    iteration: int = 0

state = AgentState(
    user_query="Explain quantum RAG"
)

print(state)

In [ ]:
state.retrieved_documents.append(
    "Paper about quantum-kernel retrieval"
)

state.tool_calls.append("arxiv_search")
state.iteration += 1

print(state)

`default_factory=list` is preferred over a plain mutable default when defining list fields.

For agent systems, explicit state models are far easier to debug than miscellaneous dictionaries mutating across ten nodes and three libraries, a software tradition humanity continues to inflict on itself.

## 24. Guardrails for agent actions

Pydantic can constrain action arguments before execution.

In [ ]:
class EmailAction(BaseModel):
    recipient: str
    subject: str = Field(min_length=1, max_length=200)
    body: str = Field(min_length=1)
    send_immediately: bool = False

email_action = EmailAction(
    recipient="researcher@example.com",
    subject="Meeting request",
    body="Would you be available next week?",
    send_immediately=False
)

print(email_action)

Your application can then add policy logic such as:

```python
if not action.send_immediately:
    require_human_confirmation()
```

Pydantic validates structure. It does **not** replace permissions, security, or human confirmation.

## 25. Model-level validation

Sometimes one field depends on another.

In [ ]:
from pydantic import model_validator

class DateRange(BaseModel):
    start_year: int
    end_year: int

    @model_validator(mode="after")
    def check_range(self):
        if self.end_year < self.start_year:
            raise ValueError(
                "end_year must be greater than or equal to start_year"
            )
        return self

valid_range = DateRange(start_year=2020, end_year=2026)
print(valid_range)

In [ ]:
try:
    DateRange(start_year=2026, end_year=2020)
except ValidationError as e:
    print(e)

# Part III: Mini AI Agent

We will now combine the ideas in a small mock agent.

Architecture:

```text
User Query
    |
    v
Tool Call Dictionary
    |
    v
Pydantic Input Validation
    |
    v
Search Tool
    |
    v
Pydantic Output Validation
    |
    v
Structured Final Answer
```

In [ ]:
class SearchInput(BaseModel):
    query: str
    max_results: int = Field(default=3, ge=1, le=10)

class SearchOutput(BaseModel):
    results: list[str]

class FinalAnswer(BaseModel):
    answer: str
    sources_used: int

## 26. Mock search tool

In [ ]:
def mock_search(search_input: SearchInput) -> SearchOutput:

    all_results = [
        f"Result 1 for {search_input.query}",
        f"Result 2 for {search_input.query}",
        f"Result 3 for {search_input.query}"
    ]

    return SearchOutput(
        results=all_results[:search_input.max_results]
    )

## 27. Simulate an LLM tool call

In [ ]:
raw_llm_tool_call = {
    "query": "quantum RAG",
    "max_results": 2
}

search_input = SearchInput.model_validate(
    raw_llm_tool_call
)

print(search_input)

## 28. Execute the tool

In [ ]:
search_output = mock_search(search_input)
print(search_output)

## 29. Create a final structured answer

In [ ]:
final_answer = FinalAnswer(
    answer=(
        "Quantum RAG combines retrieval-augmented generation "
        "with quantum or quantum-inspired retrieval methods."
    ),
    sources_used=len(search_output.results)
)

print(final_answer)

The key lesson is not this fake search function.

It is the contract between every step:

```text
untrusted / uncertain data
        |
        v
Pydantic model
        |
        v
trusted program logic
```

# Part IV: Connection to Agent Frameworks

## 30. LangChain-style structured output

Modern LLM frameworks often accept Pydantic models directly.

Conceptually:

```python
class Answer(BaseModel):
    answer: str
    confidence: float

structured_llm = llm.with_structured_output(Answer)

result = structured_llm.invoke(
    "Explain quantum entanglement"
)
```

Instead of returning a raw string, the framework can return an `Answer` object.

Exact APIs can vary by version and model provider.

## 31. Tool schemas

A tool can be described with a Pydantic model:

```python
class CalculatorInput(BaseModel):
    expression: str
```

The framework can transform that model into JSON Schema and expose it to the LLM.

The model then knows the tool expects something like:

```json
{
  "expression": "2 + 2"
}
```

## 32. Agent state in graph-based systems

A graph-based agent may maintain data such as:

```python
class AgentState(BaseModel):
    messages: list[str]
    retrieved_context: list[str]
    next_action: str | None
    iteration: int
```

Each graph node reads state and produces an update.

```text
START
  |
  v
Planner
  |
  +--> Search Node
  |
  +--> Tool Node
  |
  +--> Answer Node
```

Pydantic gives those nodes explicit data contracts.

# Part V: Design Advice

## 33. Where to use Pydantic in AI agents

Good use cases include:

1. LLM structured output
2. tool arguments
3. tool results
4. agent decisions
5. agent state
6. API requests and responses
7. configuration
8. validation and guardrails
9. inter-agent messages
10. schema generation

## 34. What Pydantic does not solve

Pydantic validates structure and constraints. It does not guarantee truth.

For example:

```json
{
  "mass_of_higgs": 9000.0
}
```

is structurally valid as a float, while scientifically wrong.

Pydantic does not automatically prevent:

- hallucinations
- false facts
- poor reasoning
- unsafe permissions
- bad tool choices
- flawed agent architecture

You still need application logic, verification, permissions, and sometimes human review.

## 35. Strong schemas are better than vague schemas

Weak:

In [ ]:
class WeakAgentOutput(BaseModel):
    data: dict

Stronger:

In [ ]:
class StrongAgentOutput(BaseModel):
    answer: str
    confidence: float = Field(ge=0.0, le=1.0)
    sources: list[str]
    requires_followup: bool

A strong schema makes an agent easier to:

- validate
- test
- debug
- monitor
- integrate

# Final mental model

Think of Pydantic as the **contract layer** between probabilistic AI and deterministic software.

```text
Natural Language
      |
      v
     LLM
      |
      v
Pydantic Model
      |
  +---+---+
  |       |
Valid   Invalid
  |       |
  v       v
 Tool   Retry / Reject
  |
  v
Structured Output
```

A useful summary is:

```text
LLMs generate possibilities.
Pydantic enforces structure.
```

# Exercises

### Exercise 1

Create:

```python
WeatherToolInput
```

with:

- `city: str`
- `country: str | None`
- `days: int`

Require:

```text
1 <= days <= 7
```

### Exercise 2

Create:

```python
PaperSummary
```

with:

- title
- authors
- year
- summary
- relevance score between 0 and 1

### Exercise 3

Create an enum containing:

```text
search
retrieve
calculate
answer
```

and use it inside an `AgentDecision` model.

### Exercise 4

Create a validator ensuring an arXiv query is not empty.

### Exercise 5

Build:

```text
LLM-generated dictionary
        |
        v
ToolInput
        |
        v
Python tool
        |
        v
ToolOutput
        |
        v
FinalAnswer
```

Use fake data first. Once the structure is clear, replacing the mock tool with LangChain, LangGraph, an LLM API, or a real search tool is straightforward.